# Paper classifier walkthrough

This notebook calls a locally served vLLM model through its OpenAI-compatible API. Change `model` to the name passed to, or exposed by, your vLLM server.

In [11]:
import sys
from pathlib import Path

# Support running Jupyter from either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "synth_extract").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from synth_extract.agents.classification import (
    ClassificationFailure,
    ClassificationResult,
    PaperClassifier,
)

## Configure the vLLM endpoint

In [12]:
base_url = "http://localhost:8000/v1"
api_key = "not-required"
model = "qwen3.6-27b"

# import os
# api_key = os.getenv("OPENROUTER_API_KEY")
# base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
# model = "openai/gpt-5-mini"

classifier = PaperClassifier(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=60.0,
    max_tokens=10000,
)
classifier.llm_config()

{'model': 'qwen3.6-27b',
 'base_url': 'http://localhost:8000/v1',
 'api_key_provided': True,
 'temperature': 0.0,
 'max_tokens': 10000,
 'timeout': 60.0,
 'max_retries': 0,
 'system_prompt_path': '/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/synth_extract/agents/classification/prompts/system_prompt.md',
 'user_template_path': '/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/synth_extract/agents/classification/prompts/user_template.md',
 'prompt_hash': 'aba24d329e7f3a95dc42c26830c3c904f8ae14dea7ef18661a4610c1c89fc430'}

## Inspect the request before calling the model

In [ ]:
title = "Synthesis and thermal characterization of bio-based polyesters"
abstract = (
    "A series of bio-based polyesters was synthesized by melt "
    "polycondensation. Their molecular weights and glass-transition "
    "temperatures were measured using GPC and DSC."
)

print(classifier.render_prompt(title, abstract))

## Classify the paper

In [ ]:
result = classifier.classify(title=title, abstract=abstract)
result

In [ ]:
if isinstance(result, ClassificationResult):
    print(f"Classification label: {result.label}")
elif isinstance(result, ClassificationFailure):
    print(f"Classification failed ({result.error_type}): {result.message}")

## Changing the classification prompt

Edit `synth_extract/agents/classification/prompts/system_prompt.md`, then run `classifier.reload_prompts()` before the next classification. You can also pass alternate prompt paths to `PaperClassifier`.

In [7]:
!pwd

/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract


In [8]:
db_path = "data/central_workspace.db"

In [ ]:
import pandas as pd
import sqlite3

In [9]:
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql(f"SELECT * FROM 'papers'", conn)

In [13]:
limit = 10

for row_number, (_, row) in enumerate(df.head(limit).iterrows(), start=1):
    title = "" if pd.isna(row["title"]) else str(row["title"])
    abstract = "" if pd.isna(row["abstract"]) else str(row["abstract"])

    result = classifier.classify(title=title, abstract=abstract)

    if isinstance(result, ClassificationResult):
        classification = result.label
    else:
        classification = f"FAILED ({result.error_type}): {result.message}"

    print(f"Paper {row_number}")
    print(f"Title: {title}")
    print(f"Abstract: {abstract}")
    print(f"Classification result: {classification}")
    print("-" * 80)


Paper 1
Title: Mechanisms underlying imiquimod-induced regression of basal cell carcinoma in vivo.
Abstract: BACKGROUND
Imiquimod is a local immune response modifier that has demonstrated potent antiviral and antitumor activity. It enhances innate and acquired immune responses via endogenous cytokine production and has proven efficacious in clearing superficial basal cell carcinoma (sBCC).


OBJECTIVE
To evaluate the mechanisms by which topical imiquimod treatment leads to sBCC clearance in vivo.


DESIGN
A pilot, open-label, nonrandomized study.


SETTING
Zurich, Switzerland.


PATIENTS
Six persons 18 years or older who had nonrecurrent primary tumors that had not undergone previous biopsy or treatment but were suitable for treatment by surgical excision. The tumors were located on the scalp, extremities, or trunk; had a minimum diameter of 1 cm and a maximum diameter of 2 cm; and were clinically and histologically consistent with sBCC.


INTERVENTIONS
Daily application of 5% imiquimo